# 16 — Solver Comparison: Superpixel OE with optimistix backends

Compares four optimistix solvers in `oe_engine_optx` using the superpixel
pipeline on the Sentinel-2 Wadden Sea example scene from NB09.

| Solver | Type | Notes |
|---|---|---|
| `GaussNewton` | least-squares | Default; J^T J Hessian approx |
| `LevenbergMarquardt` | least-squares | Adds damping λ·I for robustness |
| `Dogleg` | least-squares | Trust-region; interpolates GN and steepest-descent steps |
| `LBFGS` | minimiser | Full gradient history; better for large-residual problems |

**Note**: JAX JIT-compiles each solver on first call — timing includes compilation
overhead.  In a warm session (e.g. production pipeline), only the first invocation
per solver pays this cost.

In [ ]:
import os
import time
import numpy as np
import xarray as xr
import lmfit
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import cmocean.cm as cm
from skimage.segmentation import mark_boundaries

from bio_optics.inversion import oe_engine, oe_engine_optx
from bio_optics.image_processing import superpixel_engine
from bio_optics.water.reflectance import albert_mobley_jax

jax.config.update('jax_enable_x64', True)

## Load data

Same Sentinel-2 Wadden Sea scene as NB09.

In [ ]:
dataset = xr.open_dataset(os.path.join(os.getcwd(), 'example_data/S2L2A_example.nc'))

wavelengths = np.array([490, 560, 665, 705, 740, 783, 842, 865])
band_vars   = list(dataset.data_vars)[1:]

Rrs_image = np.stack(
    [dataset.isel(time=1)[v].values for v in band_vars], axis=-1
) / np.pi

n_rows, n_cols, n_obs = Rrs_image.shape
n_pixels = n_rows * n_cols
print(f'Rrs_image: {Rrs_image.shape}  ({n_pixels} pixels, {n_obs} bands)')

## Configuration

Identical to NB09: two free parameters (`f_mix_0`, `zB`) with OE priors.

In [ ]:
_LUT_NAMES   = ['const10%', 'sand', 'coral', 'CCA', 'macrophyte', 'seagrass']
bottom_types = [1, 5]   # sand + seagrass

fit_config = {
    'C_0':     dict(vary=False, value=0.5,  sigma_a=1.0, log=True),
    'C_Y':     dict(vary=False, value=0.1,  sigma_a=1.0, log=True),
    'C_Mie':   dict(vary=False, value=0.1,  sigma_a=1.0, log=True),
    'zB':      dict(vary=True,  value=0.5,  sigma_a=0.7, log=True),
    'f_mix_0': dict(vary=True,  value=0.0,  sigma_a=2.0, log=False),
}

noise       = 0.0031
n_segments  = 1200
compactness = 0.1
slic_sigma  = 2.0
k           = 4
n_components = 6

MAX_STEPS = 100   # optimistix iteration cap

PARAM_META = {
    'zB':      ('Depth z_B',          'm',     cm.deep),
    'f_mix_0': ('Mix logit (sand)',    'logit', 'RdYlGn'),
}

# --- build lmfit Parameters --------------------------------------------------
params = lmfit.Parameters()
params.add('theta_sun',  value=np.radians(30), vary=False)
params.add('theta_view', value=np.radians(0),  vary=False)
params.add('n1',         value=1.0,            vary=False)
params.add('n2',         value=1.33,           vary=False)
params.add('kappa_0',    value=1.0546,         vary=False)
params.add('C_0',   value=fit_config['C_0']['value'],   vary=fit_config['C_0']['vary'])
params.add('C_Y',   value=fit_config['C_Y']['value'],   vary=fit_config['C_Y']['vary'])
params.add('C_Mie', value=fit_config['C_Mie']['value'], vary=fit_config['C_Mie']['vary'])
for c in range(6):
    params.add(f'C_{c+1}', value=0.0, vary=False)
params.add('C_X', value=0.0, vary=False)
params.add('S',                   value=0.014,  vary=False)
params.add('S_NAP',               value=0.011,  vary=False)
params.add('lambda_0',            value=440.0,  vary=False)
params.add('K',                   value=0.0,    vary=False)
params.add('T_W',                 value=18.0,   vary=False)
params.add('T_W_0',               value=20.0,   vary=False)
params.add('a_NAP_spec_lambda_0', value=0.041,  vary=False)
params.add('bb_phy_spec',         value=0.0010, vary=False)
params.add('bb_Mie_spec',         value=0.0042, vary=False)
params.add('bb_X_spec',           value=0.0086, vary=False)
params.add('lambda_S',            value=500.0,  vary=False)
params.add('n',                   value=-1.0,   vary=False)
for i in range(6):
    params.add(f'f_{i}', value=1.0 if i == 0 else 0.0, vary=False)
    params.add(f'B_{i}', value=1/np.pi, vary=False)
params.add('f_mix_0', value=fit_config['f_mix_0']['value'], vary=True)
params.add('zB',      value=fit_config['zB']['value'],      vary=True)

sigma_a    = {n: c['sigma_a'] for n, c in fit_config.items() if c['vary']}
log_params = [n for n, c in fit_config.items() if c['vary'] and c['log']]

# --- precompute + remap bottom LUT ------------------------------------------
pre_base     = albert_mobley_jax.precompute(wavelengths)
R_b_full     = np.array(pre_base['R_b_i'])
R_b_selected = np.zeros_like(R_b_full)
for dst, src in enumerate(bottom_types):
    R_b_selected[:, dst] = R_b_full[:, src]
pre = {**pre_base, 'R_b_i': jnp.array(R_b_selected)}

all_names = list(params.keys())
f_vec     = albert_mobley_jax.make_forward_vec(all_names, pre)
setup     = oe_engine.build_inversion(params, f_vec, sigma_a, log_params=log_params)

print('Free parameters:', setup.fit_names)
print('log_params:',      log_params)

## Run — all four solvers

Each solver runs the full superpixel pipeline (SLIC → invert → back-interpolate).
SLIC segmentation is shared; only the OE inversion step differs.

JAX JIT-compiles each solver on first call; timing includes compilation.

In [ ]:
SOLVERS = ['GaussNewton', 'LevenbergMarquardt', 'Dogleg', 'LBFGS']

results_all = {}
timings     = {}

for solver in SOLVERS:
    print(f'Running {solver} ...', end=' ', flush=True)
    t0 = time.perf_counter()
    res = superpixel_engine.invert_image_superpixel(
        Rrs_image, setup, noise,
        n_segments=n_segments,
        compactness=compactness,
        sigma=slic_sigma,
        k=k,
        n_components=n_components,
        invert_fn=oe_engine_optx.invert_image,
        store_sp_results=True,
        store_chi2_spectral=True,
        store_y_hat=True,
        solver=solver,
        max_steps=MAX_STEPS,
    )
    timings[solver]     = time.perf_counter() - t0
    results_all[solver] = res
    print(f'{timings[solver]:.1f}s')

## Timing comparison

In [ ]:
n_segs = len(results_all[SOLVERS[0]]['sp_counts'])

print(f'\n{"Solver":<22s} {"Time":>8s}  {"ms/superpixel":>14s}')
print('-' * 50)
for solver in SOLVERS:
    t = timings[solver]
    print(f'{solver:<22s} {t:>8.1f} s  {1000*t/n_segs:>14.1f} ms')

fig, ax = plt.subplots(figsize=(7, 3))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
bars = ax.bar(SOLVERS, [timings[s] for s in SOLVERS], color=colors, alpha=0.85)
ax.bar_label(bars, fmt='%.1f s', padding=3, fontsize=10)
ax.set_ylabel('Wall-clock time [s]  (includes JIT compilation)')
ax.set_title(f'Solver timing — {n_segs} superpixels, max_steps={MAX_STEPS}')
ax.set_ylim(0, max(timings.values()) * 1.25)
plt.tight_layout()
plt.show()

## Fit quality — chi2_spectral maps

`chi2_spectral = mean((Rrs_obs − Rrs_hat)²)` — plain MSE without noise weighting.
Lower is better; differences between solvers reflect convergence quality.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

chi2_all = {s: results_all[s]['chi2_spectral'] for s in SOLVERS}
vmax = np.nanpercentile(np.concatenate([c.ravel() for c in chi2_all.values()]), 98)

for ax, solver, color in zip(axes.ravel(), SOLVERS, colors):
    chi2 = chi2_all[solver]
    med  = float(np.nanmedian(chi2))
    im   = ax.imshow(chi2, cmap='RdYlGn_r', origin='upper', vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax, shrink=0.85, label='chi2_spectral')
    ax.set_title(f'{solver}\nmedian={med:.2e}')
    ax.axis('off')

plt.suptitle('chi2_spectral per solver (lower = better fit)', y=1.01)
plt.tight_layout()
plt.show()

## Fit quality — chi2 distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# chi2_spectral
for solver, color in zip(SOLVERS, colors):
    chi2 = results_all[solver]['chi2_spectral'].ravel()
    axes[0].hist(chi2[np.isfinite(chi2)], bins=60, density=True, histtype='step',
                 color=color, lw=2, label=f'{solver} (med={np.nanmedian(chi2):.2e})')
axes[0].set_xlabel('chi2_spectral')
axes[0].set_ylabel('Density')
axes[0].set_title('chi2_spectral distribution')
axes[0].legend(fontsize=8)

# chi2_calibrated (OE chi2 × N)
for solver, color in zip(SOLVERS, colors):
    chi2 = results_all[solver]['chi2_calibrated'].ravel()
    axes[1].hist(chi2[np.isfinite(chi2)], bins=60, range=(0, 5), density=True,
                 histtype='step', color=color, lw=2,
                 label=f'{solver} (med={np.nanmedian(chi2):.2f})')
axes[1].axvline(1.0, color='k', ls='--', lw=1.5, label='target χ²=1')
axes[1].set_xlabel('chi2_calibrated (OE χ² × N pixels)')
axes[1].set_ylabel('Density')
axes[1].set_title('chi2_calibrated distribution')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Convergence — n_steps distributions

Fewer steps = faster convergence per superpixel.  
Saturating at `max_steps` indicates the solver did not converge.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# distribution
for solver, color in zip(SOLVERS, colors):
    ns = results_all[solver]['n_steps'].ravel()
    ns = ns[np.isfinite(ns)]
    axes[0].hist(ns, bins=range(0, MAX_STEPS + 2), density=True, histtype='step',
                 color=color, lw=2,
                 label=f'{solver} (med={np.nanmedian(ns):.0f}, max={int(ns.max())})')
axes[0].axvline(MAX_STEPS, color='k', ls='--', lw=1.5, label=f'cap={MAX_STEPS}')
axes[0].set_xlabel('n_steps per pixel')
axes[0].set_ylabel('Density')
axes[0].set_title('Solver iteration count distribution')
axes[0].legend(fontsize=8)

# map of n_steps for first solver
ref_solver = SOLVERS[0]
im = axes[1].imshow(results_all[ref_solver]['n_steps'], cmap='plasma',
                    origin='upper', vmin=0, vmax=MAX_STEPS)
plt.colorbar(im, ax=axes[1], label='n_steps')
axes[1].set_title(f'n_steps map — {ref_solver}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Retrieved parameters — maps per solver

In [ ]:
fit_names = results_all[SOLVERS[0]]['fit_names']

for param, (label, unit, cmap) in PARAM_META.items():
    if param not in fit_names:
        continue
    i = fit_names.index(param)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    maps = [results_all[s]['x_hat'][..., i] for s in SOLVERS]
    vmin, vmax = np.nanpercentile(np.concatenate([m.ravel() for m in maps]), [2, 98])

    for ax, solver, m, color in zip(axes, SOLVERS, maps, colors):
        im = ax.imshow(m, cmap=cmap, origin='upper', vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.85, label=unit)
        ax.set_title(solver)
        ax.axis('off')

    plt.suptitle(f'{label} [{unit}]', y=1.01)
    plt.tight_layout()
    plt.show()

## Difference maps — each solver vs GaussNewton

In [ ]:
ref = SOLVERS[0]

for param, (label, unit, _) in PARAM_META.items():
    if param not in fit_names:
        continue
    i = fit_names.index(param)

    ref_map  = results_all[ref]['x_hat'][..., i]
    other    = [s for s in SOLVERS if s != ref]
    diffs    = [results_all[s]['x_hat'][..., i] - ref_map for s in other]
    dabs     = np.nanpercentile(np.abs(np.concatenate([d.ravel() for d in diffs])), 98)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, solver, diff, color in zip(axes, other, diffs, colors[1:]):
        rmsd = np.sqrt(np.nanmean(diff ** 2))
        im   = ax.imshow(diff, cmap='RdBu_r', origin='upper', vmin=-dabs, vmax=dabs)
        plt.colorbar(im, ax=ax, shrink=0.85, label=unit)
        ax.set_title(f'{solver} − {ref}\nRMSD={rmsd:.4g}')
        ax.axis('off')

    plt.suptitle(f'{label} [{unit}] — difference vs {ref}', y=1.01)
    plt.tight_layout()
    plt.show()

## Spot-check — observed vs fitted spectra

Pick the superpixel with the highest `chi2_spectral` (worst fit) and the median
superpixel, then compare observed vs simulated spectra across all four solvers.

In [ ]:
# Recover superpixel spectra from the labels of the first run (SLIC is deterministic)
ref     = SOLVERS[0]
labels  = results_all[ref]['labels']
sp_spectra, sp_counts = superpixel_engine.aggregate_superpixels(Rrs_image, labels)

sp_chi2_ref = results_all[ref]['sp_results']['chi2_spectral']

# pick worst and median superpixels (among water superpixels with finite chi2)
valid_sp    = np.isfinite(sp_chi2_ref) & np.isfinite(sp_spectra).all(axis=-1)
valid_idx   = np.where(valid_sp)[0]
ranked      = valid_idx[np.argsort(sp_chi2_ref[valid_idx])]
idx_worst   = ranked[-1]
idx_median  = ranked[len(ranked) // 2]

fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey='row')

for row, (sp_idx, title) in enumerate([
    (idx_worst,  'worst-fit superpixel'),
    (idx_median, 'median-fit superpixel'),
]):
    obs = sp_spectra[sp_idx]

    for col, (solver, color) in enumerate(zip(SOLVERS, colors)):
        ax = axes[row, col]

        sp_res = results_all[solver]['sp_results']
        y_hat  = sp_res['y_hat'][sp_idx]           # (n_obs,) — simulated spectrum
        chi2   = float(sp_res['chi2_spectral'][sp_idx])

        ax.plot(wavelengths, obs,   'k-',  lw=2,   label='observed')
        ax.plot(wavelengths, y_hat, '--',  lw=2,   color=color, label='fitted')
        ax.fill_between(wavelengths, obs, y_hat, alpha=0.15, color=color)
        ax.set_title(f'{solver}\nchi2_sp={chi2:.2e}')
        ax.set_xlabel('λ [nm]')
        if col == 0:
            ax.set_ylabel('Rrs [sr⁻¹]')
            ax.text(0.02, 0.97, title, transform=ax.transAxes,
                    va='top', ha='left', fontsize=9, style='italic')
        if row == 0 and col == 0:
            ax.legend(fontsize=8)

plt.suptitle('Observed vs fitted spectra — worst and median superpixels', y=1.01)
plt.tight_layout()
plt.show()

## Summary table

In [ ]:
print(f'\n{"Solver":<22s} {"Time":>8s}  {"med chi2_sp":>12s}  {"med chi2_cal":>13s}  {"med n_steps":>11s}')
print('-' * 75)
for solver in SOLVERS:
    r    = results_all[solver]
    t    = timings[solver]
    c_sp = float(np.nanmedian(r['chi2_spectral']))
    c_cl = float(np.nanmedian(r['chi2_calibrated']))
    ns   = float(np.nanmedian(r['n_steps']))
    print(f'{solver:<22s} {t:>8.1f} s  {c_sp:>12.3e}  {c_cl:>13.3f}  {ns:>11.0f}')